# Predicting the 2026 FIFA World Cup Champions

An updated take on the original `worldcup_pred_2022.ipynb` notebook for the **2026 FIFA World Cup** (USA / Canada / Mexico).

### What's new in this notebook

1. **Better ML model** — we replace the original `RandomForestClassifier` with **XGBoost**, plus richer feature engineering (recent form, rolling ELO‑style rating, FIFA‑rank deltas, goal differential, squad scores).
2. **Microsoft Foundry LLM** — we plug an LLM from the [Azure AI Foundry model catalog](https://ai.azure.com/explore/models) into the pipeline to generate qualitative match commentary and to act as a tie‑breaker for close knockout fixtures.  The default model is **`gpt-4o`**, but the same code works for any chat‑completions model in the catalog (`gpt-4o-mini`, `o3-mini`, `Phi-4`, `Llama-3.3-70B-Instruct`, `Mistral-Large-2411`, …) — just change `FOUNDRY_MODEL`.
3. **Full 2026 bracket simulation** — we simulate the new 48‑team / 12‑group format end‑to‑end and crown a champion.

> The Foundry calls are *optional*: if no Azure credentials are configured the notebook falls back to a pure‑ML prediction and still runs end‑to‑end.

## 1. Install / import dependencies

In [ ]:
# Run once. XGBoost + the Azure AI Inference SDK are the only new dependencies.
%pip install --quiet pandas numpy scikit-learn xgboost matplotlib seaborn azure-ai-inference azure-identity

In [ ]:
import os
import json
import math
import random
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, log_loss, classification_report
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## 2. Configure the Microsoft Foundry model

Pick any model from the [Foundry catalog](https://ai.azure.com/explore/models) — `gpt-4o` is a strong default. To enable LLM features set the two environment variables below before running the notebook:

```bash
export AZURE_AI_ENDPOINT="https://<your-foundry-project>.services.ai.azure.com/models"
export AZURE_AI_API_KEY="<your-key>"
```

If they are missing the notebook silently degrades to ML‑only predictions.

In [ ]:
FOUNDRY_MODEL = "gpt-4o"  # try: gpt-4o-mini, o3-mini, Phi-4, Llama-3.3-70B-Instruct, Mistral-Large-2411
AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_KEY = os.getenv("AZURE_AI_API_KEY")

foundry_client = None
try:
    from azure.ai.inference import ChatCompletionsClient
    from azure.ai.inference.models import SystemMessage, UserMessage
    from azure.core.credentials import AzureKeyCredential

    if AZURE_AI_ENDPOINT and AZURE_AI_API_KEY:
        foundry_client = ChatCompletionsClient(
            endpoint=AZURE_AI_ENDPOINT,
            credential=AzureKeyCredential(AZURE_AI_API_KEY),
        )
        print(f"\u2705 Foundry client ready using model '{FOUNDRY_MODEL}'")
    else:
        print("\u26a0\ufe0f  AZURE_AI_ENDPOINT / AZURE_AI_API_KEY not set \u2014 LLM features disabled.")
except ImportError:
    print("\u26a0\ufe0f  azure-ai-inference not installed \u2014 LLM features disabled.")


def ask_foundry(system: str, user: str, max_tokens: int = 300) -> str | None:
    """Send a prompt to the configured Foundry chat model. Returns None on failure."""
    if foundry_client is None:
        return None
    try:
        resp = foundry_client.complete(
            model=FOUNDRY_MODEL,
            messages=[SystemMessage(content=system), UserMessage(content=user)],
            temperature=0.3,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()
    except Exception as exc:
        print(f"Foundry call failed: {exc}")
        return None

## 3. Load and clean the historical match data

In [ ]:
matches = pd.read_csv("international_matches.csv", parse_dates=["date"])
print(f"Loaded {len(matches):,} matches between {matches['date'].min().date()} and {matches['date'].max().date()}")
matches.head(3)

## 4. Feature engineering

On top of the original FIFA rank / squad‑score features we compute, **per team and per match date**:

* `recent_form_5` — average points (Win=3, Draw=1, Lose=0) in the last 5 matches
* `goal_diff_5` — average goal differential in the last 5 matches
* `elo` — running ELO rating (K=24, start 1500)
* `rank_diff`, `points_diff`, `elo_diff` — head‑to‑head deltas

In [ ]:
df = matches.sort_values("date").reset_index(drop=True).copy()

# ---- rolling form & goal differential -------------------------------------------------
def expand_team_view(d: pd.DataFrame) -> pd.DataFrame:
    home = d[["date", "home_team", "home_team_score", "away_team_score", "home_team_result"]].rename(
        columns={"home_team": "team", "home_team_score": "gf", "away_team_score": "ga", "home_team_result": "result"}
    )
    away = d[["date", "away_team", "away_team_score", "home_team_score", "home_team_result"]].rename(
        columns={"away_team": "team", "away_team_score": "gf", "home_team_score": "ga"}
    )
    away["result"] = away["home_team_result"].map({"Win": "Lose", "Lose": "Win", "Draw": "Draw"})
    away = away.drop(columns=["home_team_result"])
    return pd.concat([home, away], ignore_index=True).sort_values("date")

team_view = expand_team_view(df)
team_view["points"] = team_view["result"].map({"Win": 3, "Draw": 1, "Lose": 0})
team_view["gd"] = team_view["gf"] - team_view["ga"]
team_view["form_5"] = team_view.groupby("team")["points"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
team_view["gd_5"] = team_view.groupby("team")["gd"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())

form_lookup = team_view.set_index(["team", "date"])[["form_5", "gd_5"]]

def lookup_form(team: str, date) -> tuple[float, float]:
    try:
        row = form_lookup.loc[(team, date)]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return float(row["form_5"] or 1.0), float(row["gd_5"] or 0.0)
    except KeyError:
        return 1.0, 0.0

df[["home_form_5", "home_gd_5"]] = df.apply(lambda r: lookup_form(r["home_team"], r["date"]), axis=1, result_type="expand")
df[["away_form_5", "away_gd_5"]] = df.apply(lambda r: lookup_form(r["away_team"], r["date"]), axis=1, result_type="expand")

In [ ]:
# ---- running ELO rating ---------------------------------------------------------------
elo = defaultdict(lambda: 1500.0)
K = 24
home_elo, away_elo = [], []
for _, row in df.iterrows():
    ht, at = row["home_team"], row["away_team"]
    rh, ra = elo[ht], elo[at]
    home_elo.append(rh); away_elo.append(ra)
    eh = 1 / (1 + 10 ** ((ra - rh) / 400))
    if row["home_team_result"] == "Win":
        sh = 1.0
    elif row["home_team_result"] == "Draw":
        sh = 0.5
    else:
        sh = 0.0
    elo[ht] = rh + K * (sh - eh)
    elo[at] = ra + K * ((1 - sh) - (1 - eh))
df["home_elo"] = home_elo
df["away_elo"] = away_elo
final_elo = dict(elo)  # save for prediction time
print("Top‑10 teams by ELO at end of dataset:")
pd.Series(final_elo).sort_values(ascending=False).head(10)

In [ ]:
# ---- final feature matrix -------------------------------------------------------------
feature_cols = [
    "home_team_fifa_rank", "away_team_fifa_rank",
    "home_team_total_fifa_points", "away_team_total_fifa_points",
    "home_team_goalkeeper_score", "away_team_goalkeeper_score",
    "home_team_mean_defense_score", "away_team_mean_defense_score",
    "home_team_mean_offense_score", "away_team_mean_offense_score",
    "home_team_mean_midfield_score", "away_team_mean_midfield_score",
    "home_form_5", "away_form_5", "home_gd_5", "away_gd_5",
    "home_elo", "away_elo",
]

data = df.dropna(subset=feature_cols + ["home_team_result"]).copy()
data["rank_diff"] = data["away_team_fifa_rank"] - data["home_team_fifa_rank"]
data["points_diff"] = data["home_team_total_fifa_points"] - data["away_team_total_fifa_points"]
data["elo_diff"] = data["home_elo"] - data["away_elo"]
feature_cols += ["rank_diff", "points_diff", "elo_diff"]

X = data[feature_cols]
le = LabelEncoder().fit(["Lose", "Draw", "Win"])
y = le.transform(data["home_team_result"])
print(f"Training rows: {len(X):,}  |  Features: {len(feature_cols)}")

## 5. Train the XGBoost model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)
print(f"Accuracy : {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred, average='weighted'):.3f}")
print(f"Log‑loss : {log_loss(y_test, proba):.3f}")
print("\nClassification report:")
print(classification_report(y_test, pred, target_names=le.classes_))

In [ ]:
# Feature importance
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(8, 6))
imp.plot(kind="barh", color="steelblue")
plt.title("XGBoost feature importance")
plt.tight_layout(); plt.show()

## 6. A reusable `predict_match()` helper

For prediction time we use each team's **latest known** FIFA rank, squad scores and ELO.

In [ ]:
latest = (
    pd.concat([
        df[["date", "home_team", "home_team_fifa_rank", "home_team_total_fifa_points",
            "home_team_goalkeeper_score", "home_team_mean_defense_score",
            "home_team_mean_offense_score", "home_team_mean_midfield_score",
            "home_form_5", "home_gd_5"]].rename(columns=lambda c: c.replace("home_team_", "").replace("home_", "")),
        df[["date", "away_team", "away_team_fifa_rank", "away_team_total_fifa_points",
            "away_team_goalkeeper_score", "away_team_mean_defense_score",
            "away_team_mean_offense_score", "away_team_mean_midfield_score",
            "away_form_5", "away_gd_5"]].rename(columns=lambda c: c.replace("away_team_", "").replace("away_", "")),
    ])
    .sort_values("date")
    .groupby("team", as_index=False)
    .last()
    .set_index("team")
)

def team_row(team: str) -> dict:
    if team not in latest.index:
        raise ValueError(f"Unknown team '{team}'.")
    r = latest.loc[team]
    return {
        "fifa_rank": r["fifa_rank"], "fifa_points": r["total_fifa_points"],
        "gk": r["goalkeeper_score"], "def": r["mean_defense_score"],
        "off": r["mean_offense_score"], "mid": r["mean_midfield_score"],
        "form": r["form_5"], "gd": r["gd_5"],
        "elo": final_elo.get(team, 1500.0),
    }

def build_features(home: str, away: str) -> pd.DataFrame:
    h, a = team_row(home), team_row(away)
    row = {
        "home_team_fifa_rank": h["fifa_rank"], "away_team_fifa_rank": a["fifa_rank"],
        "home_team_total_fifa_points": h["fifa_points"], "away_team_total_fifa_points": a["fifa_points"],
        "home_team_goalkeeper_score": h["gk"], "away_team_goalkeeper_score": a["gk"],
        "home_team_mean_defense_score": h["def"], "away_team_mean_defense_score": a["def"],
        "home_team_mean_offense_score": h["off"], "away_team_mean_offense_score": a["off"],
        "home_team_mean_midfield_score": h["mid"], "away_team_mean_midfield_score": a["mid"],
        "home_form_5": h["form"], "away_form_5": a["form"],
        "home_gd_5": h["gd"], "away_gd_5": a["gd"],
        "home_elo": h["elo"], "away_elo": a["elo"],
        "rank_diff": a["fifa_rank"] - h["fifa_rank"],
        "points_diff": h["fifa_points"] - a["fifa_points"],
        "elo_diff": h["elo"] - a["elo"],
    }
    return pd.DataFrame([row])[feature_cols]

def predict_match(home: str, away: str) -> dict:
    """Return win / draw / lose probabilities for `home` vs `away`."""
    probs = model.predict_proba(build_features(home, away))[0]
    return {cls: float(p) for cls, p in zip(le.classes_, probs)}

predict_match("Argentina", "France")

## 7. Hybrid prediction — XGBoost + Foundry LLM

For knockout fixtures we feed the team stats *and* the model probabilities into the Foundry chat model and ask it to (a) write short analyst commentary and (b) act as a tie‑breaker when the win/lose gap is small.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert football (soccer) analyst. "
    "Given two national teams, their stats, and a machine‑learning model's win/draw/lose "
    "probabilities, write 2‑3 sentences of analysis and then output a STRICT JSON line of the "
    "form: {\"winner\": \"<team name>\", \"confidence\": <0‑1>}."
)

def llm_pick_winner(home: str, away: str, probs: dict) -> tuple[str, str]:
    """Returns (winner, commentary). Falls back to the ML pick if Foundry is unavailable."""
    ml_winner = home if probs["Win"] >= probs["Lose"] else away
    if foundry_client is None:
        return ml_winner, f"[ML‑only] Picking {ml_winner} based on XGBoost probabilities {probs}."

    h, a = team_row(home), team_row(away)
    prompt = (
        f"Match: {home} (home) vs {away} (away)\n"
        f"{home}: FIFA rank {h['fifa_rank']:.0f}, ELO {h['elo']:.0f}, recent form {h['form']:.2f}, gd {h['gd']:.2f}, "
        f"squad scores GK/DEF/MID/OFF = {h['gk']:.1f}/{h['def']:.1f}/{h['mid']:.1f}/{h['off']:.1f}.\n"
        f"{away}: FIFA rank {a['fifa_rank']:.0f}, ELO {a['elo']:.0f}, recent form {a['form']:.2f}, gd {a['gd']:.2f}, "
        f"squad scores GK/DEF/MID/OFF = {a['gk']:.1f}/{a['def']:.1f}/{a['mid']:.1f}/{a['off']:.1f}.\n"
        f"XGBoost probabilities (from {home}'s point of view): {probs}.\n"
        f"Knockout fixture — no draw allowed. Who wins?"
    )
    text = ask_foundry(SYSTEM_PROMPT, prompt) or ""
    winner = ml_winner
    for line in reversed(text.splitlines()):
        line = line.strip().strip("`")
        if line.startswith("{") and line.endswith("}"):
            try:
                obj = json.loads(line)
                cand = obj.get("winner", "").strip()
                if cand in (home, away):
                    winner = cand
                break
            except json.JSONDecodeError:
                continue
    return winner, text or f"[ML‑only] {ml_winner}"

def knockout(home: str, away: str, verbose: bool = True) -> str:
    probs = predict_match(home, away)
    winner, commentary = llm_pick_winner(home, away, probs)
    if verbose:
        print(f"\n\u26bd  {home} vs {away}\n   probs={ {k: round(v, 3) for k, v in probs.items()} } \u2192 \u2b50 {winner}")
        if commentary:
            print("   " + commentary.replace("\n", "\n   "))
    return winner

## 8. The 2026 World Cup — 48 teams, 12 groups

We use a plausible list of 48 qualifiers (final field will be confirmed in 2026).  Group winners and runners‑up plus the eight best third‑placed teams advance to a 32‑team knockout.

In [ ]:
GROUPS = {
    "A": ["Mexico", "Belgium", "Ecuador", "South Korea"],
    "B": ["Canada", "Croatia", "Senegal", "Australia"],
    "C": ["USA", "Netherlands", "Japan", "Egypt"],
    "D": ["Argentina", "Denmark", "Ivory Coast", "Iran"],
    "E": ["France", "Switzerland", "Tunisia", "New Zealand"],
    "F": ["Brazil", "Uruguay", "Cameroon", "Saudi Arabia"],
    "G": ["England", "Serbia", "Morocco", "Qatar"],
    "H": ["Germany", "Poland", "Ghana", "Costa Rica"],
    "I": ["Spain", "Sweden", "Algeria", "Panama"],
    "J": ["Portugal", "Wales", "Nigeria", "Jamaica"],
    "K": ["Italy", "Austria", "Mali", "Paraguay"],
    "L": ["Colombia", "Norway", "South Africa", "Chile"],
}

# Sanity‑check: drop any team that isn't in our dataset, swap in a fallback.
FALLBACK = ["Peru", "Turkey", "Czech Republic", "Romania", "Slovakia", "Hungary", "Greece", "Scotland"]
missing = [t for grp in GROUPS.values() for t in grp if t not in latest.index]
if missing:
    print(f"Teams missing from dataset, substituting: {missing}")
    for g, teams in GROUPS.items():
        GROUPS[g] = [t if t in latest.index else FALLBACK.pop(0) for t in teams]

In [ ]:
def simulate_group(group_name: str, teams: list[str]) -> pd.DataFrame:
    table = {t: {"P": 0, "W": 0, "D": 0, "L": 0, "PTS": 0, "xGD": 0.0} for t in teams}
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            h, a = teams[i], teams[j]
            p = predict_match(h, a)
            outcome = max(p, key=p.get)
            table[h]["P"] += 1; table[a]["P"] += 1
            if outcome == "Win":
                table[h]["W"] += 1; table[h]["PTS"] += 3; table[a]["L"] += 1
            elif outcome == "Lose":
                table[a]["W"] += 1; table[a]["PTS"] += 3; table[h]["L"] += 1
            else:
                table[h]["D"] += 1; table[a]["D"] += 1
                table[h]["PTS"] += 1; table[a]["PTS"] += 1
            edge = p["Win"] - p["Lose"]
            table[h]["xGD"] += edge; table[a]["xGD"] -= edge
    return (
        pd.DataFrame.from_dict(table, orient="index")
        .sort_values(["PTS", "xGD"], ascending=False)
        .assign(Group=group_name)
    )

group_tables = {g: simulate_group(g, teams) for g, teams in GROUPS.items()}
for g, tbl in group_tables.items():
    print(f"\nGroup {g}\n{'-'*30}")
    print(tbl[["P", "W", "D", "L", "PTS", "xGD"]].round(2))

In [ ]:
# Top two from each group + 8 best third‑placed teams = 32 teams in knockouts
firsts = [tbl.index[0] for tbl in group_tables.values()]
seconds = [tbl.index[1] for tbl in group_tables.values()]
thirds = sorted(
    [(tbl.index[2], tbl.iloc[2]["PTS"], tbl.iloc[2]["xGD"]) for tbl in group_tables.values()],
    key=lambda x: (x[1], x[2]),
    reverse=True,
)[:8]
thirds = [t[0] for t in thirds]

advancing = firsts + seconds + thirds
print(f"{len(advancing)} teams advancing to the Round of 32:")
print(", ".join(advancing))

In [ ]:
def run_bracket(teams: list[str]) -> str:
    rounds = ["Round of 32", "Round of 16", "Quarter‑finals", "Semi‑finals", "Final"]
    bracket = teams.copy()
    for r in rounds:
        print(f"\n=========== {r} ===========")
        next_round = []
        for i in range(0, len(bracket), 2):
            h, a = bracket[i], bracket[i + 1]
            next_round.append(knockout(h, a))
        bracket = next_round
    return bracket[0]

random.shuffle(advancing)
champion = run_bracket(advancing)
print("\n" + "\u2605" * 40)
print(f"  Predicted 2026 FIFA World Cup champion: {champion}")
print("\u2605" * 40)

## 9. Ask the Foundry model for a final summary

In [ ]:
summary = ask_foundry(
    system="You are a football pundit. Be concise and engaging.",
    user=(
        f"My XGBoost + Foundry hybrid model predicts {champion} to win the 2026 FIFA World Cup. "
        "Write a punchy 4‑sentence headline + closing paragraph celebrating the pick and listing "
        "two reasons it is plausible and one reason it might be wrong."
    ),
    max_tokens=350,
)
print(summary or f"(LLM disabled) The model picks {champion}.")

---
### Next steps / ideas

* Swap `FOUNDRY_MODEL` for `o3-mini` or `Phi-4` and compare bracket outcomes.
* Run the knockout bracket as a Monte‑Carlo simulation (1000× with sampled outcomes) instead of greedy argmax.
* Fine‑tune the FIFA‑specific reasoning prompt with few‑shot examples of historical upsets.
* Replace XGBoost with a small neural net or LightGBM and ensemble them.